In [ ]:
from collections import defaultdict, deque
from pathlib import Path
import re

import pandas as pd

# ---- User settings ---------------------------------------------------------
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'code' else Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data'
KRAKEN_DB = Path('/mnt/d/kraken2DB/k2_core_nt_2025')
OUTPUT_DIR = DATA_ROOT / 'kraken_parsed'

# Set either TARGET_TAXID or TARGET_NAME. If TARGET_TAXID is set, it wins.
TARGET_TAXID = 7149
TARGET_NAME = None  # example: 'Coleoptera'

# If True, skip .kraken files that do not yet have the matching .report file.
# Keep True while Kraken2 is still running, because .kraken can exist before .report.
REQUIRE_MATCHING_REPORT = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NAMES_DMP = KRAKEN_DB / 'names.dmp'
NODES_DMP = KRAKEN_DB / 'nodes.dmp'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

Project root: /mnt/c/Users/mftor/Documents/metabarcoding
Data root: /mnt/c/Users/mftor/Documents/metabarcoding/data
Output directory: /mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed


In [2]:
def slugify(value):
    value = str(value).strip().lower()
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_') or 'target_taxon'


def path_metadata(path):
    path = Path(path)
    parts = path.parts
    barcode_index = next((i for i, part in enumerate(parts) if part.startswith('barcode')), None)

    barcode = parts[barcode_index] if barcode_index is not None else None
    merge_folder = parts[barcode_index - 1] if barcode_index and barcode_index > 0 else None
    barcode_folder = str(Path(*parts[:barcode_index + 1])) if barcode_index is not None else None

    return {
        'merge_folder': merge_folder,
        'barcode': barcode,
        'barcode_folder': barcode_folder,
        'source_file': str(path),
        'source_name': path.name,
    }


def read_names(names_dmp):
    taxid_to_name = {}
    name_to_taxids = defaultdict(list)

    with open(names_dmp, encoding='utf-8') as handle:
        for line in handle:
            fields = [field.strip() for field in line.split('|')]
            taxid = int(fields[0])
            name = fields[1]
            name_class = fields[3]
            if name_class == 'scientific name':
                taxid_to_name[taxid] = name
                name_to_taxids[name.lower()].append(taxid)

    return taxid_to_name, name_to_taxids


def read_nodes(nodes_dmp):
    taxid_to_parent = {}
    taxid_to_rank = {}
    children_by_parent = defaultdict(list)

    with open(nodes_dmp, encoding='utf-8') as handle:
        for line in handle:
            fields = [field.strip() for field in line.split('|')]
            taxid = int(fields[0])
            parent_taxid = int(fields[1])
            rank = fields[2]
            taxid_to_parent[taxid] = parent_taxid
            taxid_to_rank[taxid] = rank
            if taxid != parent_taxid:
                children_by_parent[parent_taxid].append(taxid)

    return taxid_to_parent, taxid_to_rank, children_by_parent


def resolve_taxid(target_taxid, target_name, taxid_to_name, name_to_taxids):
    if target_taxid is not None:
        target_taxid = int(target_taxid)
        if target_taxid not in taxid_to_name:
            raise ValueError(f'TARGET_TAXID {target_taxid} was not found in names.dmp')
        return target_taxid

    if not target_name:
        raise ValueError('Set TARGET_TAXID or TARGET_NAME')

    matches = name_to_taxids.get(target_name.lower(), [])
    if not matches:
        raise ValueError(f'TARGET_NAME {target_name!r} was not found as a scientific name in names.dmp')
    if len(matches) > 1:
        print(f'TARGET_NAME {target_name!r} matched multiple taxIDs {matches}; using {matches[0]}')
    return matches[0]


def descendant_taxids(root_taxid, children_by_parent):
    descendants = set()
    queue = deque([root_taxid])

    while queue:
        taxid = queue.popleft()
        if taxid in descendants:
            continue
        descendants.add(taxid)
        queue.extend(children_by_parent.get(taxid, []))

    return descendants

LINEAGE_RANKS = ['kingdom', 'class', 'order', 'family', 'genus']


def lineage_for_taxid(taxid, taxid_to_parent, taxid_to_rank, taxid_to_name):
    lineage = {rank: None for rank in LINEAGE_RANKS}

    if pd.isna(taxid):
        return lineage

    taxid = int(taxid)
    if taxid <= 0 or taxid not in taxid_to_parent:
        return lineage

    seen = set()
    current = taxid

    while current not in seen and current in taxid_to_parent:
        seen.add(current)
        rank = taxid_to_rank.get(current)
        if rank in lineage and lineage[rank] is None:
            lineage[rank] = taxid_to_name.get(current)

        parent = taxid_to_parent.get(current)
        if parent is None or parent == current:
            break
        current = parent

    return lineage


def lineage_table_for_taxids(taxids, taxid_to_parent, taxid_to_rank, taxid_to_name):
    rows = []
    for value in taxids.dropna().unique():
        taxid = int(value)
        if taxid <= 0:
            continue
        row = {'taxonID': taxid}
        row.update(lineage_for_taxid(taxid, taxid_to_parent, taxid_to_rank, taxid_to_name))
        rows.append(row)

    return pd.DataFrame(rows, columns=['taxonID'] + LINEAGE_RANKS)



In [3]:
# Load NCBI taxonomy and build the target clade taxID set.
taxid_to_name, name_to_taxids = read_names(NAMES_DMP)
taxid_to_parent, taxid_to_rank, children_by_parent = read_nodes(NODES_DMP)

target_taxid = resolve_taxid(TARGET_TAXID, TARGET_NAME, taxid_to_name, name_to_taxids)
target_name = taxid_to_name[target_taxid]
target_rank = taxid_to_rank.get(target_taxid)
target_slug = slugify(target_name)
target_descendant_taxids = descendant_taxids(target_taxid, children_by_parent)

target_taxonomy = pd.DataFrame({'taxonID': sorted(target_descendant_taxids)})
target_taxonomy['taxonName'] = target_taxonomy['taxonID'].map(taxid_to_name)
target_taxonomy['taxonRank'] = target_taxonomy['taxonID'].map(taxid_to_rank)

taxid_to_taxon_name = taxid_to_name
taxid_to_taxon_rank = taxid_to_rank

print(f'Target: {target_name} (taxID {target_taxid}, rank {target_rank})')
print(f'Target + daughter taxIDs: {len(target_descendant_taxids):,}')
target_taxonomy.head()

Target: Coleoptera (taxID 7041, rank order)
Target + daughter taxIDs: 95,314


,taxonID,taxonName,taxonRank
0,7041,Coleoptera,order
1,7042,Curculionidae,family
2,7043,Anthonomus,genus
3,7044,Anthonomus grandis,species
4,7045,Sitophilus,genus


In [4]:
# Discover Kraken outputs across all barcode folders.
all_kraken_files = sorted(DATA_ROOT.glob('**/barcode*/**/*.kraken'))
all_report_files = sorted(DATA_ROOT.glob('**/barcode*/**/*.report'))
report_files_set = set(all_report_files)

missing_report_files = sorted(
    Path(str(path)[:-len('.kraken')] + '.report')
    for path in all_kraken_files
    if Path(str(path)[:-len('.kraken')] + '.report') not in report_files_set
)

kraken_files = [
    path for path in all_kraken_files
    if (not REQUIRE_MATCHING_REPORT)
    or Path(str(path)[:-len('.kraken')] + '.report') in report_files_set
]

print(f'Kraken files found: {len(all_kraken_files)}')
print(f'Kraken files selected: {len(kraken_files)}')
print(f'Report files found: {len(all_report_files)}')
print(f'Kraken files missing matching report: {len(missing_report_files)}')

for path in missing_report_files[:10]:
    print('missing report:', path)

Kraken files found: 42
Kraken files selected: 40
Report files found: 40
Kraken files missing matching report: 2
missing report: /mnt/c/Users/mftor/Documents/metabarcoding/data/merged/barcode57/filtered/combined/combined_nogroup_unique.fasta.report
missing report: /mnt/c/Users/mftor/Documents/metabarcoding/data/merged/barcode59/filtered/combined/combined_nogroup_unique.fasta.report


In [5]:
def read_kraken_file(path):
    columns = ['status', 'readID', 'taxonID', 'length', 'LCA_list']
    df = pd.read_csv(path, sep='\t', header=None, names=columns, dtype={'taxonID': 'Int64'})
    if df.empty:
        return df

    metadata = path_metadata(path)
    for key, value in metadata.items():
        df[key] = value

    return df


kraken_tables = [read_kraken_file(path) for path in kraken_files]
kraken_reads = (
    pd.concat([df for df in kraken_tables if not df.empty], ignore_index=True)
    if kraken_tables
    else pd.DataFrame(columns=['status', 'readID', 'taxonID', 'length', 'LCA_list'])
)

if not kraken_reads.empty:
    kraken_reads['taxonID'] = pd.to_numeric(kraken_reads['taxonID'], errors='coerce').astype('Int64')
    # Kraken2 assigns taxonID; organism_name/rank are looked up from NCBI names.dmp/nodes.dmp.
    kraken_reads['organism_name'] = kraken_reads['taxonID'].map(taxid_to_name)
    kraken_reads['organism_rank'] = kraken_reads['taxonID'].map(taxid_to_rank)

    lineage_by_taxid = lineage_table_for_taxids(
        kraken_reads['taxonID'],
        taxid_to_parent,
        taxid_to_rank,
        taxid_to_name,
    )
    kraken_reads = kraken_reads.merge(lineage_by_taxid, on='taxonID', how='left')
    kraken_reads['is_target_or_descendant'] = kraken_reads['taxonID'].isin(target_descendant_taxids)
else:
    kraken_reads['organism_name'] = pd.Series(dtype='object')
    kraken_reads['organism_rank'] = pd.Series(dtype='object')
    for rank in LINEAGE_RANKS:
        kraken_reads[rank] = pd.Series(dtype='object')
    kraken_reads['is_target_or_descendant'] = pd.Series(dtype='bool')

target_reads = kraken_reads[kraken_reads['is_target_or_descendant']].copy()

print(f'Parsed Kraken read rows: {len(kraken_reads):,}')
print(f'{target_name} + daughter read rows: {len(target_reads):,}')
kraken_reads.head()

Parsed Kraken read rows: 247,736
Coleoptera + daughter read rows: 16,612


,status,readID,taxonID,length,LCA_list,merge_folder,barcode,barcode_folder,source_file,source_name,organism_name,organism_rank,kingdom,class,order,family,genus,is_target_or_descendant
0,U,3229,0,817,0:4 131567:1 1:8 0:770,merged,barcode49,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,combined_0_unique.fasta.kraken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,U,3354,0,833,131567:3 1:8 0:18 29875:5 0:750 29875:5 0:10,merged,barcode49,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,combined_0_unique.fasta.kraken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,U,4064,0,756,0:2 131567:1 1:8 0:18 29875:5 0:688,merged,barcode49,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,combined_0_unique.fasta.kraken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,U,4119,0,814,0:2 1:8 0:18 29875:5 0:747,merged,barcode49,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,combined_0_unique.fasta.kraken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,U,5330,0,833,573:2 0:2 131567:1 1:8 0:774 29875:5 0:7,merged,barcode49,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,combined_0_unique.fasta.kraken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [6]:
# Save complete read table plus filtered target-clade table.
all_reads_path = OUTPUT_DIR / 'all_kraken_reads.csv'
target_reads_path = OUTPUT_DIR / f'{target_slug}_kraken_reads.csv'
target_taxids_path = OUTPUT_DIR / f'{target_slug}_taxids.csv'
missing_reports_path = OUTPUT_DIR / 'kraken_files_missing_reports.csv'

kraken_reads.to_csv(all_reads_path, index=False)
target_reads.to_csv(target_reads_path, index=False)
target_taxonomy.to_csv(target_taxids_path, index=False)
pd.DataFrame({'missing_report_file': [str(path) for path in missing_report_files]}).to_csv(missing_reports_path, index=False)

print('Saved:')
print(all_reads_path)
print(target_reads_path)
print(target_taxids_path)
print(missing_reports_path)

Saved:
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/all_kraken_reads.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/coleoptera_kraken_reads.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/coleoptera_taxids.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/kraken_files_missing_reports.csv


In [7]:
# Optional summaries for quick inspection.
target_read_counts_by_taxon = (
    target_reads
    .groupby(['taxonID', 'organism_name', 'organism_rank'], dropna=False)
    .size()
    .reset_index(name='read_count')
    .sort_values('read_count', ascending=False)
)

target_read_counts_by_barcode = (
    target_reads
    .groupby(['merge_folder', 'barcode', 'taxonID', 'organism_name', 'organism_rank'], dropna=False)
    .size()
    .reset_index(name='read_count')
    .sort_values(['merge_folder', 'barcode', 'read_count'], ascending=[True, True, False])
)

target_read_counts_by_taxon.to_csv(OUTPUT_DIR / f'{target_slug}_read_counts_by_taxon.csv', index=False)
target_read_counts_by_barcode.to_csv(OUTPUT_DIR / f'{target_slug}_read_counts_by_barcode.csv', index=False)

print(f'Observed target/daughter taxIDs: {target_read_counts_by_taxon.shape[0]:,}')
target_read_counts_by_taxon.head(25)

Observed target/daughter taxIDs: 1,083


,taxonID,organism_name,organism_rank,read_count
788,1971854,Ptenidium turgidum,species,1132
635,1553578,Myllaena minuta,species,732
679,1587295,Euryusa castanoptera,species,692
374,577917,Cis micans,species,650
614,1553213,Clambus lohsei,species,493
278,347338,Dinaraea aequata,species,446
701,1588484,Stenus ludyi,species,412
476,1002105,Xyleborus cryptographus,species,396
28,41084,Polyphaga,suborder,380
279,347343,Euplectus,genus,378
